# 01 — The data, and what it looks like before you model itStage 1, first half. The goal is to know this dataset well enough that nothing later surprises you.By the end you should be able to say, with numbers: how many trials and classes there are, how sparse the activity is, how much trial durations vary, and what the two preprocessing choices (`bin_ms`, `t_max_ms`) actually do to the data.

In [ ]:
%load_ext autoreload%autoreload 2import numpy as npimport matplotlib.pyplot as pltfrom compbio2026 import data, plottingplotting.apply_style()rng = np.random.default_rng(2026)

## LoadThe first call downloads ~200 MB and caches it under `data/`. If you already have the file from the 2025 repository, pass `path=` and skip the download.

In [ ]:
shd = data.load("train")print(f"{len(shd)} trials, {len(np.unique(shd.labels))} classes, {len(np.unique(shd.speaker))} speakers")print("first five words:", shd.words[:5])

## What is actually in a trialThe data is stored as an event list: for each trial, spike times in seconds and the channel each spike came from. Nothing is on a grid yet.

In [ ]:
i = 0print("spike times :", shd.times[i][:8], "...")print("channels    :", shd.units[i][:8], "...")print("n spikes    :", len(shd.times[i]))print("duration    :", shd.times[i].max(), "s")print("label       :", shd.labels[i], "=", data.DIGIT_KEYS[shd.labels[i]])

## Class balance and trial durationTwo sanity checks that change what you do next. Class imbalance changes your chance level; duration variability is the reason `t_max_ms` is a decision rather than a default.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))counts = np.bincount(shd.labels, minlength=20)axes[0].bar(np.arange(20), counts, color=plotting.ACCENT, linewidth=0)axes[0].set_xticks(np.arange(20))axes[0].set_xticklabels(data.DIGIT_KEYS, rotation=90, fontsize=7)axes[0].set_ylabel("trials")axes[0].set_title("Class balance")durations = np.array([t.max() for t in shd.times])axes[1].hist(durations, bins=50, color=plotting.ACCENT, linewidth=0)axes[1].set_xlabel("Trial duration (s)")axes[1].set_ylabel("trials")axes[1].set_title("Duration varies with speaker")fig.tight_layout()print(f"duration: median {np.median(durations):.2f} s, 5-95% {np.percentile(durations, 5):.2f}-{np.percentile(durations, 95):.2f} s")

**Look at that right-hand panel before choosing `t_max_ms`.** Truncating loses the tail of long trials; zero-padding invents silence in short ones. Both distort the geometry you are about to measure. Whatever you pick, check your result survives a different choice.

## RastersThe structure is visible by eye. Whether it is *usable* is the question the rest of the project answers.

In [ ]:
X, y, t = data.build_design_matrix(shd, bin_ms=10.0, t_max_ms=800.0)print("X:", X.shape, "(trials, channels, bins)")fig, axes = plt.subplots(2, 2, figsize=(9, 6), sharex=True, sharey=True)for ax, word in zip(axes.ravel(), ["seven", "six", "three", "one"]):    idx = np.flatnonzero(shd.labels == data.DIGIT_KEYS.index(word))    plotting.raster(X, idx[0], ax=ax, bin_ms=10.0)    ax.set_title(f'"{word}"')fig.tight_layout()

Channel index is **tonotopic** — low index to high index maps monotonically onto frequency, because position along the modelled basilar membrane is set by frequency. The diagonal sweeps you can see are formant transitions moving along that axis. Any analysis that permutes channels destroys this; sometimes that is exactly the control you want.

## Population statistics

In [ ]:
rates = data.rates(X, bin_ms=10.0)            # (trials, channels), spikes/sfig, axes = plt.subplots(1, 3, figsize=(13, 3.4))axes[0].hist(rates.mean(axis=0), bins=60, color=plotting.ACCENT, linewidth=0)axes[0].set_xlabel("Mean rate (spikes/s)")axes[0].set_ylabel("channels")axes[0].set_title("Rate distribution across channels")sparsity = (X > 0).mean(axis=(0, 2))axes[1].plot(sparsity, color=plotting.ACCENT)axes[1].set_xlabel("Channel (tonotopic)")axes[1].set_ylabel("P(spike in a bin)")axes[1].set_title("Sparsity is not uniform")axes[2].plot(t, X.mean(axis=(0, 1)) * 100, color=plotting.ACCENT)axes[2].set_xlabel("Time (ms)")axes[2].set_ylabel("Mean spikes/bin (x100)")axes[2].set_title("Population time course")fig.tight_layout()

## Trial-to-trial variabilityHow reproducible is a digit? Correlate trials of the same digit against trials of different digits.

In [ ]:
from scipy.spatial.distance import pdist, squareformsub = rng.choice(len(shd), size=400, replace=False)Xf = data.flatten(X[sub])Xf = Xf / (np.linalg.norm(Xf, axis=1, keepdims=True) + 1e-9)sim = 1 - squareform(pdist(Xf, metric="cosine"))same = y[sub][:, None] == y[sub][None, :]iu = np.triu_indices(len(sub), k=1)fig, ax = plt.subplots(figsize=(5, 3.4))ax.hist(sim[iu][same[iu]], bins=40, alpha=0.7, label="same digit", density=True, linewidth=0)ax.hist(sim[iu][~same[iu]], bins=40, alpha=0.7, label="different digit", density=True, linewidth=0)ax.set_xlabel("Cosine similarity")ax.set_ylabel("density")ax.legend()print(f"same    {sim[iu][same[iu]].mean():.3f}")print(f"differ  {sim[iu][~same[iu]].mean():.3f}")

The overlap between those two distributions is a preview of how hard the decoding problem is. If they separated cleanly, nothing later in this project would be interesting.

## The choice that is not a defaultBin width sets how much spike timing survives. Cramer et al. showed classifiers with no access to timing plateau near 60 %, while temporally aware ones reach ~85 %. See what binning does to the data before you commit to a value.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)for ax, b in zip(axes, [1.0, 5.0, 20.0, 50.0]):    Xb, yb, tb = data.build_design_matrix(shd, bin_ms=b, t_max_ms=800.0, trials=np.arange(200))    ax.imshow(Xb[0], aspect="auto", origin="lower", cmap="Greys",              extent=[0, 800, 0, Xb.shape[1]])    ax.set_title(f"bin = {b:g} ms")    ax.set_xlabel("Time (ms)")axes[0].set_ylabel("Channel")fig.tight_layout()

---## Exercises1. **Speakers.** Are some speakers systematically louder (more spikes)? If so, a classifier could learn the speaker instead of the digit. Check, and decide what to do about it.2. **Language.** Compare the English digits (labels 0–9) to their German counterparts (10–19). Are the "same" digits in the two languages more similar to each other than to other digits?3. **A surrogate control.** Destroy temporal structure by shuffling spike times within each trial, keeping channel counts fixed. Re-run the similarity analysis. How much of the same/different separation survives? Keep this surrogate — it is the right null for several later analyses.4. **Pick your `t_max_ms` and defend it in two sentences.**